In [0]:
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import yfinance as yf
from pyspark.sql.functions import *

In [0]:
%sql
CREATE DATABASE financial_risk_db;
USE financial_risk_db;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7026608787441096>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE DATABASE financial_risk_db;\nUSE financial_risk_db;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:205, in SqlMagic.sql(self, line, cell)
    198 except BaseException as e:
    199     self.dr

In [0]:
import yfinance as yf
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
ticker = yf.Ticker("AAPL")

df = ticker.history(period="30d", interval="1d")

df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-02-02 00:00:00-05:00,259.786917,270.237131,258.967677,269.757599,73913400,0.0,0.0
2026-02-03 00:00:00-05:00,268.948351,271.625839,267.359811,269.228088,64394700,0.0,0.0
2026-02-04 00:00:00-05:00,272.035451,278.689229,272.035451,276.231506,90545700,0.0,0.0
2026-02-05 00:00:00-05:00,277.869995,279.238709,272.974582,275.652069,52977400,0.0,0.0
2026-02-06 00:00:00-05:00,276.860920,280.647386,276.671095,277.859985,50453400,0.0,0.0


In [0]:
df = df.reset_index()

spark_df = spark.createDataFrame(df)

spark_df.printSchema()

spark_df.show(5)
spark_df = spark_df.withColumn("Date", col("Date").cast("timestamp"))

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock Splits: double (nullable = true)

+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+
|               Date|              Open|              High|               Low|             Close|  Volume|Dividends|Stock Splits|
+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+
|2026-02-02 05:00:00| 259.7869174093499| 270.2371306501278| 258.9676766447768| 269.7575988769531|73913400|      0.0|         0.0|
|2026-02-03 05:00:00| 268.9483513556569| 271.6258386480508| 267.3598109321867|269.22808837890625|64394700|      0.0|         0.0|
|2026-02-04 05:00:00|   272.03545112075|

In [0]:
spark_df = spark_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

spark_df = spark_df.withColumn(
    "ingestion_date",
    current_date()
)

spark_df.show(5)

+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+--------------------+--------------+
|               Date|              Open|              High|               Low|             Close|  Volume|Dividends|Stock Splits| ingestion_timestamp|ingestion_date|
+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+--------------------+--------------+
|2026-02-02 05:00:00| 259.7869174093499| 270.2371306501278| 258.9676766447768| 269.7575988769531|73913400|      0.0|         0.0|2026-03-17 12:08:...|    2026-03-17|
|2026-02-03 05:00:00| 268.9483513556569| 271.6258386480508| 267.3598109321867|269.22808837890625|64394700|      0.0|         0.0|2026-03-17 12:08:...|    2026-03-17|
|2026-02-04 05:00:00|   272.03545112075|278.68922850453845|   272.03545112075|276.23150634765625|90545700|      0.0|         0.0|2026-03-17 12:08:...|    2026-03-17|
|202

In [0]:
spark_df = spark_df.withColumnRenamed("Stock Splits", "Stock_Splits")

In [0]:
spark_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("ingestion_date") \
    .saveAsTable("bronze_stock_data")

In [0]:
%sql
SELECT * FROM bronze_stock_data LIMIT 5;

Date,Open,High,Low,Close,Volume,Dividends,Stock_Splits,ingestion_timestamp,ingestion_date
2026-02-02T05:00:00.000Z,259.7869174093499,270.2371306501278,258.9676766447768,269.7575988769531,73913400,0.0,0.0,2026-03-17T12:09:06.093Z,2026-03-17
2026-02-03T05:00:00.000Z,268.9483513556569,271.6258386480508,267.3598109321867,269.22808837890625,64394700,0.0,0.0,2026-03-17T12:09:06.093Z,2026-03-17
2026-02-04T05:00:00.000Z,272.03545112075,278.68922850453845,272.03545112075,276.23150634765625,90545700,0.0,0.0,2026-03-17T12:09:06.093Z,2026-03-17
2026-02-05T05:00:00.000Z,277.86999494352693,279.2387093202658,272.97458180817284,275.6520690917969,52977400,0.0,0.0,2026-03-17T12:09:06.093Z,2026-03-17
2026-02-06T05:00:00.000Z,276.8609202349588,280.6473855638201,276.67109542368036,277.8599853515625,50453400,0.0,0.0,2026-03-17T12:09:06.093Z,2026-03-17


In [0]:
from pyspark.sql.functions import max

try:
    last_date = spark.sql("""
        SELECT MAX(Date) as max_date 
        FROM bronze_stock_data
    """).collect()[0]["max_date"]

    print("Last loaded date:", last_date)

except:
    last_date = None
    print("First run — no table yet")

Last loaded date: 2026-03-16 04:00:00


In [0]:
import yfinance as yf

ticker = yf.Ticker("AAPL")

df = ticker.history(period="60d", interval="1d")
df = df.reset_index()

In [0]:
spark_df = spark.createDataFrame(df)

from pyspark.sql.functions import col

spark_df = spark_df.withColumn("Date", col("Date").cast("timestamp"))

In [0]:
spark_df.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock Splits: double (nullable = true)



In [0]:
if last_date:
    spark_df = spark_df.filter(col("Date") > last_date)
    print("Filtering new records only")
else:
    print("No previous data — full load")

Filtering new records only


In [0]:
from pyspark.sql.functions import *

In [0]:
spark_df = spark_df.withColumn("ingestion_timestamp", current_timestamp())
spark_df = spark_df.withColumn("ingestion_date", current_date())

In [0]:
spark_df = spark_df.toDF(*[col.replace(" ", "_") for col in spark_df.columns])

In [0]:
spark_df.write.format("delta") \
    .mode("append") \
    .partitionBy("ingestion_date") \
    .saveAsTable("bronze_stock_data")

In [0]:
import re

def clean_column(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name)

spark_df = spark_df.toDF(*[clean_column(col) for col in spark_df.columns])

In [0]:
spark_df = spark_df.dropDuplicates(["Date"])

In [0]:
%sql
SELECT COUNT(*) FROM bronze_stock_data;

COUNT(*)
30
